# Mapa de Desarrollo Economico - Barrio Obrero (oficial)

Dimension **Desarrollo Economico** del ITT (peso 15%). Integra los negocios
(registros de Camara de Comercio: categoria, sector economico, tamano, empleo,
CIIU) y las sedes educativas de contexto. Estandar Ciudad Paraiso: paleta
Okabe-Ito, capas base Google, mascara, slider de calor, panel colapsable con
serie anual y boton 'Exportar lamina (deck)'.

Salida: `mapa_html/Barrio_obrero/Barrio_obrero_actual/mapa_desarrollo_economico_barrio_obrero.html`.

## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','rasterio','pillow'],check=False)
    print('Colab: dependencias instaladas')
else:
    print('Entorno local (uv).')


## 1. Entorno, rutas y paleta

In [ ]:
import os, json
import numpy as np, pandas as pd, geopandas as gpd
import folium
# Paleta Okabe-Ito
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_AMARILLO="#F0E442"
OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"; OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"; LIMITE="#1B4F8A"; CRS_WEB=4326; CRS_M=3116
def _ruta(*cands):
    for p in cands:
        if os.path.isfile(p) or os.path.isdir(p): return os.path.abspath(p)
    return None
def _bo(*parts):
    return _ruta(os.path.join("..","data","Barrio_Obrero_geojson",*parts),
                 os.path.join("data","Barrio_Obrero_geojson",*parts),
                 os.path.join("/content","data","Barrio_Obrero_geojson",*parts))
def _dd(*parts):
    return _ruta(os.path.join("..","data",*parts),
                 os.path.join("data",*parts),
                 os.path.join("/content","data",*parts))
ZONA_DIR="barrio_obrero_completo"
DATADIR=_bo(ZONA_DIR)
IPM_PATH=_bo("IPM_Barrio_Obrero_completo.geojson")
POLY_PATH=_bo("Geojson_Barrio_Obrero.geojson")
LIMITE_PATH=_bo(ZONA_DIR,"00_limite_barrio_obrero_completo.geojson")
CAI_PATH=_bo(ZONA_DIR,"5_Entorno_Urbano","CAI_MECAL_CALI_OBRERO.geojson")
NDVI_DIR=_bo(ZONA_DIR,"5_Entorno_Urbano","NDVI_Barrio_Obrero","NDVI_Barrio_Obrero")
DEFICIT_PATH=_dd("5_Dimension_Entorno_Urbano","sec_vivienda","DEF_HAB_SISBEN.shp")
if DATADIR is None or POLY_PATH is None:
    raise FileNotFoundError("Faltan datos base de Barrio Obrero")
def _en(d,pref):
    if d is None: return None
    for f in os.listdir(d):
        if f.startswith(pref) and f.endswith(".geojson"): return os.path.join(d,f)
    return None
POLIGONO_ZONA=gpd.read_file(POLY_PATH).to_crs(CRS_WEB)
_zg=POLIGONO_ZONA.geometry.union_all()
_c=_zg.centroid
_b=POLIGONO_ZONA.total_bounds
BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
# salida
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try: os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: pass
_raiz=os.path.dirname(OUT_DIR.rstrip(os.sep))
MAPAS_DIR=os.path.abspath(os.path.join(_raiz,"mapa_html","Barrio_obrero","Barrio_obrero_actual"))
os.makedirs(MAPAS_DIR,exist_ok=True)
ZOOM=15
print("Zona:",len(POLIGONO_ZONA),"poligono | centro",round(_c.y,5),round(_c.x,5))
print("Salida:",MAPAS_DIR)


## 2. Carga de negocios (limpieza de encoding) y educacion

In [ ]:
def _limpiar(s):
    if s is None: return ""
    s=str(s)
    _rep={"REPARACI?N":"REPARACION","GESTI?N":"GESTION","CONSTRUCCI?N":"CONSTRUCCION",
          "VEH?CULOS":"VEHICULOS","CIENT?FICAS":"CIENTIFICAS","T?CNICAS":"TECNICAS",
          "ALCOH?LICAS":"ALCOHOLICAS","ART?CULOS":"ARTICULOS","FERRETER?A":"FERRETERIA",
          "FABRICACI?N":"FABRICACION","MET?LICOS":"METALICOS","PEQUE?A":"PEQUENA"}
    for k,v in _rep.items(): s=s.replace(k,v)
    return s.replace("?","").strip()
def _norm_tam(s):
    s=_limpiar(s).upper()
    if "MICRO" in s: return "Micro"
    if "PEQUE" in s: return "Pequena"
    if "MEDIAN" in s: return "Mediana"
    if "GRANDE" in s: return "Grande"
    return "Sin dato"
_pn=_en(DATADIR,"09_desarrollo_economico_negocios")
if _pn:
    neg=gpd.read_file(_pn).to_crs(CRS_WEB)
    neg=neg[neg.geometry.notna() & neg.geom_type.str.contains("Point")].copy()
    neg["cat"]=neg["categoria"].map(_limpiar) if "categoria" in neg.columns else ""
    neg["sector"]=neg["nombre_sec"].map(_limpiar) if "nombre_sec" in neg.columns else ""
    neg["ciiu"]=neg["desc_ciiu"].map(_limpiar) if "desc_ciiu" in neg.columns else ""
    neg["tam"]=neg["tamano957"].map(_norm_tam) if "tamano957" in neg.columns else "Sin dato"
    neg["empleo"]=pd.to_numeric(neg["personal_o"],errors="coerce").fillna(0).astype(int) if "personal_o" in neg.columns else 0
    neg["lon"]=neg.geometry.x; neg["lat"]=neg.geometry.y
else:
    neg=gpd.GeoDataFrame(columns=["cat","sector","ciiu","tam","empleo","lon","lat","geometry"],crs=CRS_WEB)
NEG_TOTAL=int(len(neg)); EMPLEO_TOTAL=int(neg["empleo"].sum()) if NEG_TOTAL else 0
TAM_ORDEN=["Grande","Mediana","Pequena","Micro","Sin dato"]
TAM_CONT={t:int((neg["tam"]==t).sum()) for t in TAM_ORDEN}
SECTOR_CONT=neg["sector"].value_counts().head(7).to_dict() if NEG_TOTAL else {}
# educacion
_pe=_en(DATADIR,"10_educacion_sedes")
edu=None
if _pe:
    edu=gpd.read_file(_pe).to_crs(CRS_WEB)
    edu=edu[edu.geometry.notna()]
EDU_N=int(len(edu)) if edu is not None else 0
print("Negocios:",NEG_TOTAL,"| empleo total:",EMPLEO_TOTAL,"| por tamano:",TAM_CONT,"| sedes educativas:",EDU_N)

## 3. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 4. Mapa base (estandar Ciudad Paraiso)

In [ ]:
import folium
from folium.plugins import HeatMap
m_eco=folium.Map(location=[_c.y,_c.x],zoom_start=ZOOM,tiles=None,control_scale=True,
             zoom_snap=0.25,zoom_delta=0.1,max_zoom=22)
m_eco.get_root().width='100%'; m_eco.get_root().height='100%'
m_eco.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
    ".leaflet-top,.leaflet-bottom{z-index:1000}"
    "#eco-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
    "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
    "#heat-intensity-ctrl{z-index:1001}</style>"))
folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',attr='Google',name='Google Streets',show=True,max_zoom=22,max_native_zoom=20).add_to(m_eco)
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',attr='Esri',name='Esri Satelite',show=False,max_zoom=22,max_native_zoom=20).add_to(m_eco)
folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',attr='Google',name='Google Satelite',show=False,max_zoom=22,max_native_zoom=20).add_to(m_eco)
_mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=='MultiPolygon' else [_zg]):
    _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({'type':'Feature','properties':{},'geometry':{'type':'Polygon','coordinates':[_mundo]+_ag}},
    name='Mascara (foco en la zona)',control=False,
    style_function=lambda f:{'fillColor':'#000','color':'#000','weight':0,'fillOpacity':0.18}).add_to(m_eco)


## 5. Negocios (categoria, tamano, sector), educacion y calor

In [ ]:
CAT_COLOR={"PERSONA NATURAL":OKI_AZUL,"PERSONA JURIDICA":OKI_VERDE}
def _ccat(c):
    return CAT_COLOR.get(str(c).strip().upper(),"#999999")
# por categoria
_cats=[c for c in neg["cat"].astype(str).str.strip().str.upper().unique() if c and c!="0"] if NEG_TOTAL else []
for cat in sorted(_cats):
    sub=neg[neg["cat"].astype(str).str.strip().str.upper()==cat]
    if len(sub)==0: continue
    fgp=folium.FeatureGroup(name="Negocios: "+cat.title()+" ("+str(len(sub))+")",show=(cat=="PERSONA NATURAL"))
    for _,rr in sub.iterrows():
        _pop=('<div style="font-family:system-ui;font-size:11px;background:%s;padding:6px 8px;border-radius:5px;max-width:240px">'
              '<b>%s</b><br>Sector: %s<br>Tamano: %s | Empleo: %d<br><span style="color:#777">%s</span></div>'
              %(BG,cat.title(),(rr["sector"] or "s/d").title(),rr["tam"],int(rr["empleo"]),(rr["ciiu"] or "")[:60]))
        folium.CircleMarker([rr["lat"],rr["lon"]],radius=3,color="#333",weight=0.4,
            fill=True,fill_color=_ccat(cat),fill_opacity=0.75,popup=folium.Popup(_pop,max_width=260)).add_to(fgp)
    fgp.add_to(m_eco)
# por tamano (capas apagadas)
TAM_COLOR={"Grande":OKI_BERMELL,"Mediana":OKI_NARANJA,"Pequena":OKI_AMARILLO,"Micro":OKI_AZUL_C,"Sin dato":"#999"}
TAM_RADIO={"Grande":7,"Mediana":6,"Pequena":5,"Micro":3,"Sin dato":3}
for t in TAM_ORDEN:
    sub=neg[neg["tam"]==t]
    if len(sub)==0: continue
    fgt=folium.FeatureGroup(name="Tamano: "+t+" ("+str(len(sub))+")",show=False)
    for _,rr in sub.iterrows():
        folium.CircleMarker([rr["lat"],rr["lon"]],radius=TAM_RADIO[t],color="#333",weight=0.4,
            fill=True,fill_color=TAM_COLOR[t],fill_opacity=0.7).add_to(fgt)
    fgt.add_to(m_eco)
# educacion
if edu is not None and len(edu)>0:
    fge=folium.FeatureGroup(name="Educacion: sedes ("+str(len(edu))+")",show=True)
    for _,rr in edu.iterrows():
        _nm=str(rr.get("nombresede",rr.get("nombreinst","Sede")))
        _dir=str(rr.get("direccion",""))
        _pt=rr.geometry.centroid if rr.geometry.geom_type!="Point" else rr.geometry
        folium.CircleMarker([_pt.y,_pt.x],radius=6,color="#333",weight=0.6,
            fill=True,fill_color=OKI_MORADO,fill_opacity=0.85,
            popup=folium.Popup('<div style="font-family:system-ui;font-size:11px;background:'+BG+';padding:6px 8px;border-radius:5px"><b>'+_nm+'</b><br>'+_dir+'</div>',max_width=240)).add_to(fge)
    fge.add_to(m_eco)
# calor de negocios
if NEG_TOTAL:
    _fgh=folium.FeatureGroup(name="Calor: negocios",show=False)
    HeatMap(neg[["lat","lon"]].values.tolist(),
            gradient={0.0:"#eff3ff",0.4:"#9ecae1",0.7:"#4292c6",1.0:"#08306b"},
            radius=18,blur=14,min_opacity=0.35).add_to(_fgh)
    _fgh.add_to(m_eco)
print("Capas de desarrollo economico agregadas.")

## 6. Limite, estacion CAI y slider de intensidad

In [ ]:
folium.GeoJson(POLIGONO_ZONA.to_json(),name='Limite Barrio Obrero',show=True,
    style_function=lambda f:{'color':LIMITE,'weight':2.5,'fill':False}).add_to(m_eco)
# Estacion de policia (CAI/MECAL) - gestion
if CAI_PATH:
    _cai=gpd.read_file(CAI_PATH).to_crs(CRS_WEB)
    _fgc=folium.FeatureGroup(name='Gestion: estacion policia (CAI)',show=True)
    for _,_r in _cai.iterrows():
        _sig=str(_r.get('SIGLA',_r.get('UNIDAD','CAI')))
        folium.Marker([_r.geometry.y,_r.geometry.x],
            tooltip='Estacion: '+_sig,
            icon=folium.Icon(color='darkblue',icon='shield',prefix='fa')).add_to(_fgc)
    _fgc.add_to(m_eco)

_slider = '''
<div id="heat-intensity-ctrl" style="position:fixed;bottom:25px;right:24px;z-index:1001;background:#F4F6F9;padding:7px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:system-ui,Segoe UI,sans-serif;font-size:11px;color:#333;max-width:190px">
  <div style="font-weight:700;margin-bottom:3px">Intensidad del calor</div>
  <input id="heat-intensity-range" type="range" min="8" max="40" value="18" style="width:150px">
  <span id="heat-intensity-val">18</span>
  <div style="font-size:9px;color:#888;margin-top:2px">Afecta las capas de mapa de calor.</div>
</div>
<script>
(function(){
  function wire(){
    var r=document.getElementById("heat-intensity-range");
    var v=document.getElementById("heat-intensity-val");
    if(!r){ setTimeout(wire,300); return; }
    var mp=window["m_eco"];
    function apply(){
      var val=parseInt(r.value,10); if(v) v.textContent=val;
      if(!mp) return;
      mp.eachLayer(function(l){
        if(l && l.setOptions && l._latlngs){ try{ l.setOptions({radius:val,blur:Math.round(val*0.8)}); l.redraw(); }catch(e){} }
      });
    }
    r.addEventListener("input",apply);
    setTimeout(apply,600);
  }
  if(document.readyState==="complete"){ wire(); }
  else{ window.addEventListener("load",wire); }
})();
</script>
'''.replace("m_eco","m_eco")
m_eco.get_root().html.add_child(folium.Element(_slider))


## 7. Leyenda y controles

In [ ]:
_items=[("Persona natural",OKI_AZUL),("Persona juridica",OKI_VERDE),("Educacion",OKI_MORADO)]
_lh=""
for _nom,_col in _items:
    _lh+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
          '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:50%%;border:1px solid #999"></i>'
          '<span style="font-size:10px">%s</span></div>'%(_col,_nom))
_ley=('<div style="position:fixed;bottom:110px;right:24px;z-index:1000;background:'+BG+';'
      'padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:system-ui,Segoe UI,sans-serif;color:'+OKI_GRIS+'">'
      '<div style="font-weight:700;font-size:11px;margin-bottom:4px">Desarrollo economico</div>'+_lh+'</div>')
m_eco.get_root().html.add_child(folium.Element(_ley))
m_eco.fit_bounds(BO_BOUNDS,padding=(30,30))
folium.LayerControl(collapsed=True,position='topright').add_to(m_eco)
print('Mapa construido.')


## 8. Panel flotante: resumen + sectores (Chart.js)

In [ ]:
_sec_items=sorted(SECTOR_CONT.items(),key=lambda kv:-kv[1])
_labels=[(_k[:24]+("..." if len(_k)>24 else "")) if _k else "Sin dato" for _k,_ in _sec_items]
_vals=[int(_v) for _,_v in _sec_items]
_chart=json.dumps({"labels":_labels,"data":_vals},ensure_ascii=False)
_resumen=('<div style="margin:4px 0 2px;font-weight:700;font-size:11px">Resumen:</div>'
          '<div style="font-size:11px;margin:1px 0">&bull; Negocios registrados: <b>%d</b></div>'
          '<div style="font-size:11px;margin:1px 0">&bull; Empleo declarado: <b>%d</b> personas</div>'
          '<div style="font-size:11px;margin:1px 0">&bull; Sedes educativas: <b>%d</b></div>'
          %(NEG_TOTAL,EMPLEO_TOTAL,EDU_N))
_resumen+='<div style="margin-top:4px;font-weight:700;font-size:11px">Por tamano:</div>'
for _t in TAM_ORDEN:
    if TAM_CONT[_t]==0: continue
    _resumen+='<div style="font-size:11px;margin:1px 0">&bull; %s: <b>%d</b></div>'%(_t,TAM_CONT[_t])
_panel=r'''
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="eco-panel" style="position:fixed;top:14px;left:14px;z-index:1001;max-width:390px;width:min(390px,44vw);max-height:84vh;overflow-y:auto;padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">Desarrollo economico - Barrio Obrero</div>
    <button id="eco-toggle" title="Minimizar / expandir" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;line-height:1;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="eco-body">
    <div style="font-size:10px;color:#666;margin:4px 0 6px 0">Negocios de Camara de Comercio por categoria, tamano y sector; sedes educativas.</div>
    __RESUMEN__
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__LIM__;outline:none">Negocios por sector economico / Ocultar</summary>
      <div style="position:relative;height:215px;width:100%;margin-top:6px"><canvas id="eco-chart"></canvas></div>
      <div id="eco-fallback" style="font-size:9px;color:#777;margin-top:3px"></div>
    </details>
  </div>
</div>
<script>
(function(){
  var D=__CHART__;
  function dibujar(){
    var cv=document.getElementById("eco-chart"); if(!cv||typeof Chart==="undefined"){ setTimeout(dibujar,300); return; }
    new Chart(cv.getContext("2d"),{type:"bar",
      data:{labels:D.labels,datasets:[{data:D.data,backgroundColor:"#0072B2",borderWidth:1}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
        scales:{x:{beginAtZero:true,ticks:{font:{size:9}},title:{display:true,text:"Cantidad",font:{size:9}}},y:{ticks:{font:{size:8}}}}}});
  }
  function initColapso(){
    var b=document.getElementById("eco-toggle"),bo=document.getElementById("eco-body");
    if(!b||!bo)return; b.addEventListener("click",function(){var o=(bo.style.display==="none");bo.style.display=o?"":"none";b.textContent=o?"_":"+";});
  }
  if(document.readyState==="complete"){setTimeout(dibujar,300);initColapso();}
  else{window.addEventListener("load",function(){setTimeout(dibujar,300);initColapso();});}
})();
</script>
'''
_panel=(_panel.replace("__GRIS__",OKI_GRIS).replace("__LIM__",LIMITE)
             .replace("__RESUMEN__",_resumen).replace("__CHART__",_chart))
m_eco.get_root().html.add_child(folium.Element(_panel))
print("Panel de desarrollo economico inyectado.")

## 9. Exportacion del HTML

In [ ]:
agregar_export(m_eco, 'Desarrollo economico - Barrio Obrero (oficial)', 'desarrollo_economico_bo')
html=os.path.join(MAPAS_DIR,'mapa_desarrollo_economico_barrio_obrero.html')
m_eco.save(html)
print('Mapa desarrollo economico exportado:', html)
m_eco